In [20]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage,AIMessage, BaseMessage
from typing import TypedDict,Literal,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
import operator 

In [21]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [22]:
llm=ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
load_dotenv()

True

In [23]:
def chat_node(state: ChatState) -> ChatState:
    response = llm.invoke(state['messages'])
    return {"messages": [response]}

In [24]:
checkpointer=MemorySaver()
graph = StateGraph(ChatState)
graph.add_node("chat_node",chat_node)
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)
chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
# response=chatbot.invoke({"messages": [HumanMessage(content="Hello, how are you?")]})
# print(response)

{'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='81570558-e6c9-4119-9ab2-58291a405cb3'), AIMessage(content="Hello! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with anything you need. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 13, 'total_tokens': 50, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DYlZ74Nwa2Rk7tBqWpSkzbzFmJVc4', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dc815-c9be-7863-9ea2-026c02abdb2d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'o

In [31]:
thread_id='thread_1'
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("Exiting chat.")
        break
    config = {"configurable": {"thread_id": thread_id}}
    response = chatbot.invoke({"messages": [HumanMessage(content=user_input)]}, config=config)
    print(f"Chatbot: {response['messages'][-1].content}")

Chatbot: Your name is Gowrish.
Exiting chat.


In [29]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='hi my name is gowrish', additional_kwargs={}, response_metadata={}, id='536b846c-0230-40bf-9b25-b580d4b56075'), AIMessage(content='Hello Gowrish! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 14, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DYluD8UHrWUynGRJceefHPbAOKQ41', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dc829-bdda-7ca3-902a-18dd15c0bde2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 11, 'total_tokens': 25, 'input_token_details': {'audio': 0, 'cache_r